In [1]:
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.feature_selection import SelectKBest, f_regression, f_classif
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, make_scorer, r2_score
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_val_score, KFold, LeaveOneGroupOut
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as torchopt
from torch.utils.data import Dataset, DataLoader, Subset


from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from joblib import Memory
import joblib

We start by importing our different datasets

In [2]:


base_dir = Path("data")
guided_dir = base_dir / "guided"
freemoves_dir = base_dir / "freemoves"


file_path_EMG = guided_dir / "guided_dataset_X.npy"
dataEMG = np.load(file_path_EMG)

file_path_HAND = guided_dir / "guided_dataset_y.npy"
dataHAND = np.load(file_path_HAND)

file_path_TestGuided = guided_dir / "guided_testset_X.npy"
GuidedTest = np.load(file_path_TestGuided)

file_path_EMG_free = freemoves_dir / "freemoves_dataset_X.npy"
dataEMGFree = np.load(file_path_EMG_free)

file_path_HAND_free = freemoves_dir / "freemoves_dataset_y.npy"
dataHANDFree = np.load(file_path_HAND_free)

file_path_TestFree = freemoves_dir / "freemoves_testset_X.npy"
FreeTest = np.load(file_path_TestFree)



We start by doing a filtering of our data. "https://www.sciencedirect.com/science/article/pii/S0021929010000631" this paper is a relevant litterature on the subject
We learn that most of the noise that occurs when recording our sEMG is located in lower frequencies. And that a high-pass filtering at 10hz is highly beneficial with diminishing performance at 20 and 30 hz 
while still being beneficial as not much revelant signal information is lost.
As the choice of the frequencie of the high-pass is muscle dependent and as we are limited by our lack of expertise. We will follow the recommendation of the paper and settle at doing a Butterworth filter with a corner frequency of 20 Hz and a slope of 12 dB/oct 
To do so we will use the butter and filtfilt functions of the scipy library

In addition, we also learn from this paper "https://iopscience.iop.org/article/10.1088/1742-6596/1237/3/032008/pdf" that most of the sEMG's energy is concentrated in the 0-500hz range 
Our sampling frenquencie being of 1024hz, twice the bandwith of the signal, by the Nyquist-Shannon theorem we can efficiently control for aliasing up to the 500hz frequencies.
Even if according to the first paper most of the noise is located in lower frequencies, since relveant information is under the 500hz threshold, we can control for higher frequencies.




In [3]:
cutoff_high = 20     # We select (in hz) the cutoff of our high-pass filter
cutoff_low = 500     # We select (in hz) the cutoff of our low-pass filter
order_high = 2       # We select the number of poles. In the Butterworth filter, each pole correspong to 6db, so we select two to get the recommended 12dp/octave
order_low = 4        # Once again we select the number of poles. In the paper the nbr of decibel/octave for the high-pass filter was 24 dB/oct, so we follow this recommendation.

b_high, a_high = butter(order_high, cutoff_high / (1024 / 2), btype='highpass')    # The butter function returns the filter coefficients a (denominator),b (numerator). And take as inupts the order and Wn, the desired cutoff frequency being the frequency at which at which the magnitude response of the filter is 1 / √2
b_low, a_low = butter(order_low, cutoff_low / (1024 / 2), btype='lowpass')

def filter_emg(signal_1d):
    high = filtfilt(b_high, a_high, signal_1d)                                     # The filtfilt function controls for distortion                         
    return filtfilt(b_low, a_low, high)

    
def filter_emg_3d(data):  
    filtered = np.empty_like(data)
    for session in range(data.shape[0]):
        for channel in range(data.shape[1]):
            filtered[session, channel] = filter_emg(data[session, channel])
    return filtered


def filter_emg_4d(data):  
    filtered = np.empty_like(data)
    for session in range(data.shape[0]):
        for window in range(data.shape[1]):
            for channel in range(data.shape[2]):
                filtered[session, window, channel] = filter_emg(data[session, window, channel])
    return filtered


dataEMG = filter_emg_3d(dataEMG)
dataEMGFree = filter_emg_3d(dataEMGFree)

GuidedTest = filter_emg_4d(GuidedTest)
FreeTest = filter_emg_4d(FreeTest)

print("Missing guided EMG:", np.isnan(dataEMG).sum())                                # We check for potential missing values
print("Missing free EMG:", np.isnan(dataEMGFree).sum())
print("Missing guided testset:", np.isnan(GuidedTest).sum())
print("Missing free testset:", np.isnan(FreeTest).sum())

n_sessions, n_electrodes, n_samples = dataEMG.shape

Missing guided EMG: 0
Missing free EMG: 0
Missing guided testset: 0
Missing free testset: 0


We do some "windowing" of our dataset by creating overlapping windows of size 500ms with a 250ms step betwwen each window.

In [4]:
ws = 500
step = 250
n_samples = 230000

def window_data(emg, hand):
    n_sessions, n_electrodes, n_samples = emg.shape
    n_joints = dataHAND.shape[1]  

    emg_windows = {}
    hand_windows = {}

    for s in range(n_sessions):
        emg_windows[s] = {}
        hand_windows[s] = {}
        for e in range(n_electrodes):
            emg_windows[s][e] = [
                emg[s, e, start:start+ws]
                for start in range(0, n_samples - ws + 1, step)
            ]
        for j in range(n_joints):
            hand_windows[s][j] = [
                hand[s, j, start:start+ws]
                for start in range(0, n_samples - ws + 1, step)
            ]
    return emg_windows, hand_windows

emg_windows, hand_windows = window_data(dataEMG, dataHAND)
emg_windows_Free, hand_windows_Free = window_data(dataEMGFree, dataHANDFree)

Now for our guided and free datasets, we organize our data so that we will predict the value t ov each window (t - 500, t).
We also define groups that will be of use in our CV strategy.

In [5]:
windows_list = []
hand_last = []
groups = []

for s in range(5):
    for w in range(918):
        groups.append(s)
        emg_stack = np.stack([emg_windows[s][e][w] for e in range(n_electrodes)], axis=0)
        windows_list.append(emg_stack)
        last_samples = [hand_windows[s][j][w][-1] for j in range(51)]
        hand_last.append(last_samples)

X = np.stack(windows_list, axis=0)
Y = np.array(hand_last)
groups = np.array(groups)

print("Guided — X.shape:", X.shape, "Y.shape:", Y.shape, "groups.shape:", groups.shape)



Guided — X.shape: (4590, 8, 500) Y.shape: (4590, 51) groups.shape: (4590,)


In [6]:
windows_list_free = []
hand_last_free = []
groups_free = []

for s in range(5):
    for w in range(918):
        groups_free.append(s)
        emg_stack = np.stack([emg_windows_Free[s][e][w] for e in range(n_electrodes)], axis=0)
        windows_list_free.append(emg_stack)
        last_samples = [hand_windows_Free[s][j][w][-1] for j in range(51)]
        hand_last_free.append(last_samples)

X_free = np.stack(windows_list_free, axis=0)
Y_free = np.array(hand_last_free)
groups_free = np.array(groups_free)

print("Free — X_free.shape:", X_free.shape, "Y_free.shape:", Y_free.shape, "groups_free.shape:", groups_free.shape)

Free — X_free.shape: (4590, 8, 500) Y_free.shape: (4590, 51) groups_free.shape: (4590,)


For our cross-validation strategy, as our windows overlap wich each others. By doing a standart randomizationn we would surely see data leakage.
To prevent this issue, we go for a Leave One Session Out cross-validation method. As our data is already organized in sessions, it makes it handy to create 5 folds.

In [7]:
logo = LeaveOneGroupOut()
logo.get_n_splits(X, Y, groups)

print("Guided:")
for i, (train_index, test_index) in enumerate(logo.split(X, Y, groups)):
    print(f"Fold {i}: Train groups={np.unique(groups[train_index])}, Test group={np.unique(groups[test_index])}")

print("\nFree:")
for i, (train_index, test_index) in enumerate(logo.split(X_free, Y_free, groups_free)):
    print(f"Fold {i}: Train groups={np.unique(groups_free[train_index])}, Test group={np.unique(groups_free[test_index])}")


Guided:
Fold 0: Train groups=[1 2 3 4], Test group=[0]
Fold 1: Train groups=[0 2 3 4], Test group=[1]
Fold 2: Train groups=[0 1 3 4], Test group=[2]
Fold 3: Train groups=[0 1 2 4], Test group=[3]
Fold 4: Train groups=[0 1 2 3], Test group=[4]

Free:
Fold 0: Train groups=[1 2 3 4], Test group=[0]
Fold 1: Train groups=[0 2 3 4], Test group=[1]
Fold 2: Train groups=[0 1 3 4], Test group=[2]
Fold 3: Train groups=[0 1 2 4], Test group=[3]
Fold 4: Train groups=[0 1 2 3], Test group=[4]


Here we define our FeatureExtractor class that will come in use in the pipelines. For the choice of our features, we use some features deemed relevant in the related litterature (https://www.sciencedirect.com/science/article/abs/pii/S0957417412001200)

In [8]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.stats import skew, kurtosis
from scipy.signal import find_peaks
from numpy.fft import rfft, rfftfreq
import pywt 

class FeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.sigma = 0.05
        self.features_name = [
            'MoyenneValAbsolue', 'RacineMoyenneCarre', 'Variance', 'StandartDeviation', 'NbrChangementSigne', 'ProportionMoyenne',
            'Median', 'Percentile10', 'Percentile90', 'EcartInterquartile',
            'Skewness', 'Kurtosis',
            'Slope', 'Intercept', 'R2',
            'AutoCorr1', 'NbrPics',
            'SpectralCentroid', 'SpectralEntropy',
            'LongueurOnde', 'ZeroCrossing', 'lopeSignChange', 'EMAV', 'EWLs'
            
        ]


    def fit(self, X, y=None):
        return self

    def transform(self, X):
        self.sigma = np.std(X)
        windows, channels, _ = X.shape
        features_number = len(self.features_name)
        transformed = np.zeros((windows, channels * features_number))

        for window in range(windows):
            for channel in range(channels):
                features = self.compute_features(X[window, channel, :])
                transformed[window,
                            channel*features_number:(channel+1)*features_number] = features
        return transformed

        

    def get_feature_names_out(self, input_features=None):
        names = []
        for ch in range(self.n_channels_):
            for fn in self.features_name:
                names.append(f"ch{ch}_{fn}")
        return np.array(names)

    def compute_features(self, x):
        t = np.arange(len(x))
        T = 0.01 * np.max(np.abs(x))

        MoyenneValAbsolue    = np.mean(np.abs(x))
        RacineMoyenneCarre   = np.sqrt(np.mean(x**2))
        Variance             = np.var(x, ddof=1)
        StandartDeviation    = np.std(x, ddof=1)
        NbrChangementSigne   = np.sum(np.diff(np.sign(x)) != 0)
        ProportionMoyenne    = np.sum(np.abs(x) > self.sigma) / len(x)

        Median               = np.median(x)
        Percentile10         = np.percentile(x, 10)
        Percentile90         = np.percentile(x, 90)
        EcartInterquartile   = Percentile90 - Percentile10
        Skewness             = skew(x)
        Kurtosis             = kurtosis(x)

        Slope, Intercept     = np.polyfit(t, x, 1)
        corr                 = np.corrcoef(t, x)[0, 1]
        R2                   = corr**2 if not np.isnan(corr) else 0.0

        AutoCorr1            = np.corrcoef(x[:-1], x[1:])[0, 1] if len(x) > 1 else 0.0
        peaks, _             = find_peaks(x)
        NbrPics              = len(peaks)

        Xf                   = np.abs(rfft(x))
        freqs                = rfftfreq(len(x), d=1.0)
        SpectralCentroid     = (freqs * Xf).sum() / (Xf.sum() + 1e-12)
        p                    = Xf / (Xf.sum() + 1e-12)
        SpectralEntropy      = -np.sum(p * np.log2(p + 1e-12))
        
        LongueurOnde = np.sum(np.abs(np.diff(x)))
        ZeroCrossing = np.sum(
        ((x[:-1] * x[1:] < 0) & (np.abs(x[:-1] - x[1:]) >= T)).astype(int))
        SlopeSignChange = np.sum(
        (((x[2:] - x[1:-1]) * (x[1:-1] - x[:-2]) < 0) & ((np.abs(x[2:] - x[1:-1]) >= T) | (np.abs(x[1:-1] - x[:-2]) >= T))).astype(int))

        L = len(x)
        weights = np.ones(L) * 0.5
        weights[int(0.2*L):int(0.8*L)] = 0.75
        EMAV = np.mean(weights * np.abs(x))

        coeffs = pywt.wavedec(x, 'bior3.3', level=4)
        EWLs = np.sum([np.sum(c**2) for c in coeffs[1:]])

        
        

        return [
            MoyenneValAbsolue, RacineMoyenneCarre, Variance, StandartDeviation, NbrChangementSigne, ProportionMoyenne,
            Median, Percentile10, Percentile90, EcartInterquartile,
            Skewness, Kurtosis,
            Slope, Intercept, R2,
            AutoCorr1, NbrPics,
            SpectralCentroid, SpectralEntropy,
            LongueurOnde, ZeroCrossing, SlopeSignChange, EMAV, EWLs
            
        ]

We now define our pipeline. To reduce dimensionality and improve computational efficiency, we use PCA.  
We also apply `StandardScaler()`, a transformer that standardizes each feature to zero mean and unit variance. This is necessary because some features are on different scales and PCA depends on scaling.  
We use a combination of three models: Decision Tree, Random Forest, and Ridge Regression.  
We chose these models for their native support of multi-output regression and because they offer a good trade-off between computational cost and performance. They are also widely used in the relevant literature, especially Random Forest.  
Finally, we configure a `GridSearchCV` to find the best combination of hyperparameters defined in our `param_grid` dictionary for each model. We run hyperparameter tuning using leave-one-session-out cross-validation, compute RMSE, and save the results.  
We also leave as comment a SelecKbest supervised filter that will select the most correlated features to ease computation on less powerful computers.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import Ridge
from joblib import Memory
import joblib

#def f_regression_multioutput(X, Y):
#   fs, ps = zip(*(f_regression(X, Y[:, i]) for i in range(Y.shape[1])))
#  fs = np.mean(fs, axis=0)
# ps = np.mean(ps, axis=0)
#return fs, ps

    
pipe = Pipeline([
    ('feat',   FeatureExtractor()),
    ('scale',  StandardScaler()),
    #    ('select', SelectKBest(score_func=f_regression_multioutput)),  
    ('pca',    PCA(n_components=0.95)), 
    ('model',  'passthrough')
])

param_grid = [                                                                   #Our dictionnary of hyperparameters for each model.                                     

  { 
#   'select__k': [15],
    'model': [DecisionTreeRegressor(random_state=42)],
    'model__max_depth': [5, 10, 15, 20],
    'model__min_samples_leaf': [1, 3, 5, 7],
  },

  {  
    
 #  'select__k': [15],
    'model': [RandomForestRegressor(random_state=42, n_jobs=2)],
    'model__n_estimators': [50, 100, 200, 250],
    'model__max_depth': [10, 20, 25],
    'model__max_features': ['sqrt', 'log2'],
  },



 {
 #   'select__k': [15],
     'model': [Ridge()],
     'model__alpha': [0.1, 1.0, 10.0]  
  }


]


gs = GridSearchCV(
    pipe,                                     #Preprocessing and pipeline  
    param_grid,                               #Our dictionary of hyperparameters
    cv=logo,                                  #Leave-One-Group-Out cross-validation
    scoring='neg_root_mean_squared_error',    #We define the scoring to be lowest RMSE is better 
    n_jobs=2,                                 #Number of parallel computation
    verbose=3                                 #Display computation progress
)



gs.fit(X, Y, groups=groups)                  #We run the fitting


results = pd.DataFrame(gs.cv_results_)                             #Save the results
results['RMSE'] = -results['mean_test_score']                    
results = results.sort_values('RMSE')                              #Organize the results
print(results[['params', 'RMSE']].to_string(index=False))
joblib.dump(gs, "gridsearch_results.pkl")                          #Save the computation for future use

In [ ]:
pipeFree = Pipeline([
    ('feat',   FeatureExtractor()),
    ('scale',  StandardScaler()),
   #('select', SelectKBest(score_func=f_regression_multioutput)), 
    ('pca',    PCA(n_components=0.95)),  
    ('model',  'passthrough')
])

param_grid = [

  { 
#    'select__k': [15],
    'model': [DecisionTreeRegressor(random_state=42)],
    'model__max_depth': [5],
    'model__min_samples_leaf': [7],
  },

  {  
    
 #   'select__k': [15],
    'model': [RandomForestRegressor(random_state=42, n_jobs=2)],
    'model__n_estimators': [200],
    'model__max_depth': [20],
    'model__max_features': ['sqrt'],
  },



 {
    #'select__k': [15],
     'model': [Ridge()],
     'model__alpha': [10.0]  
  }


]


gs_free = GridSearchCV(
    pipeFree,                                  
    param_grid,
    cv=logo,                      
    scoring='neg_root_mean_squared_error',
    n_jobs=2,
    verbose=3
)


gs_free.fit(X_free, Y_free, groups=groups_free)


results_free = pd.DataFrame(gs_free.cv_results_)
results_free['RMSE'] = -results_free['mean_test_score']
results_free = results_free.sort_values('RMSE')
print("=== Résultats GridSearch - Libre ===")
print(results_free[['params', 'RMSE']].to_string(index=False))
joblib.dump(gs_free, "gridsearch_results_free.pkl")

Fitting 5 folds for each of 3 candidates, totalling 15 fits
[CV 1/5] END model=DecisionTreeRegressor(random_state=42), model__max_depth=5, model__min_samples_leaf=7;, score=-7.095 total time=  57.7s
[CV 2/5] END model=DecisionTreeRegressor(random_state=42), model__max_depth=5, model__min_samples_leaf=7;, score=-8.499 total time=  57.9s
[CV 3/5] END model=DecisionTreeRegressor(random_state=42), model__max_depth=5, model__min_samples_leaf=7;, score=-7.414 total time=  58.3s
[CV 4/5] END model=DecisionTreeRegressor(random_state=42), model__max_depth=5, model__min_samples_leaf=7;, score=-6.725 total time=  58.4s
[CV 5/5] END model=DecisionTreeRegressor(random_state=42), model__max_depth=5, model__min_samples_leaf=7;, score=-5.919 total time= 1.0min
[CV 1/5] END model=RandomForestRegressor(n_jobs=2, random_state=42), model__max_depth=20, model__max_features=sqrt, model__n_estimators=200;, score=-6.457 total time= 1.1min
[CV 2/5] END model=RandomForestRegressor(n_jobs=2, random_state=42), mo

/home/zorbmax/ULB/ML/ML_project_classroom/pose-estimation-from-emg-signal-team-38/venv/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV 3/5] END model=RandomForestRegressor(n_jobs=2, random_state=42), model__max_depth=20, model__max_features=sqrt, model__n_estimators=200;, score=-6.674 total time= 1.1min
[CV 4/5] END model=RandomForestRegressor(n_jobs=2, random_state=42), model__max_depth=20, model__max_features=sqrt, model__n_estimators=200;, score=-6.103 total time= 1.0min
[CV 5/5] END model=RandomForestRegressor(n_jobs=2, random_state=42), model__max_depth=20, model__max_features=sqrt, model__n_estimators=200;, score=-5.590 total time= 1.1min
[CV 1/5] END .model=Ridge(), model__alpha=10.0;, score=-6.642 total time=  58.8s
[CV 2/5] END .model=Ridge(), model__alpha=10.0;, score=-7.490 total time=  58.2s
[CV 3/5] END .model=Ridge(), model__alpha=10.0;, score=-6.599 total time=  59.3s
[CV 4/5] END .model=Ridge(), model__alpha=10.0;, score=-6.505 total time=  59.5s
[CV 5/5] END .model=Ridge(), model__alpha=10.0;, score=-5.584 total time=  56.0s
=== Résultats GridSearch - Libre ===
                                    

['gridsearch_results_free.pkl']

# Neural Network

To complement our baseline models built on handcrafted features, we implemented a deep neural network using PyTorch.

We designed a Convolutional Neural Network (CNN). The network consists of:

Three convolutional blocks, each with:

- 1D Convolution layer to capture local patterns in the EMG signal per electrode.
- Batch Normalization and ReLU activation to accelerate training and improve generalization.
- AdaptiveAvgPooling layer to progressively reduce dimensionality.

Final fully connected layers, with:

- Flattening of the extracted features.
- Dense layers (512 → 128 → 51), ending in 51 outputs to match the number of target joint angles.

This architecture is inspired by prior literature in biomedical signal processing and CNNs for time-series data.

https://www.researchgate.net/publication/339471768_Rethinking_1D-CNN_for_Time_Series_Classification_A_Stronger_Baseline

https://core.ac.uk/download/617892666.pdf

We implemented Leave-One-Group-Out Cross Validation (LOGO), where each group corresponds to a session. This strategy ensures no data leakage, as overlapping windows are limited to within a session.

We assessed the neural network using:

- Fold-wise RMSE on validation sets

- Global RMSE and R² score over all folds

In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import LeaveOneGroupOut
from joblib import dump
import os
import torch.nn as nn

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")  #So that if no valid GPU is available for computation, those are made with CPU

class EMGData(Dataset):
    def __init__(self, features, labels):
        self.feats = torch.tensor(features, dtype=torch.float32)
        self.labs = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return self.feats.shape[0]

    def __getitem__(self, id):
        return self.feats[id], self.labs[id]

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.extract = nn.Sequential(
            nn.Conv1d(8,64,7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64,128,5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128,256,3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(10)
        )
        self.final_layer = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*10,512),
            nn.ReLU(),
            nn.Linear(512,128),
            nn.ReLU(),
            nn.Linear(128,51)
        )

    def forward(self, x):
        rep = self.extract(x)
        out = self.final_layer(rep)
        return out

def execute_training(net, dloader_tr, dloader_vl, E=60):
    loss_fn = nn.MSELoss()
    opt = torchopt.Adam(net.parameters(), lr=1e-3)
    sched = torchopt.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)

    net.to(DEV)
    lowest_rmse = 99999.0
    save_best = None
    bad_rounds = 0
    max_wait = 10

    for e in range(E):
        net.train()
        for x_, y_ in dloader_tr:
            x_, y_ = x_.to(DEV), y_.to(DEV)
            opt.zero_grad()
            err = loss_fn(net(x_), y_)
            err.backward()
            opt.step()

        
        net.eval()
        preds_buf, labs_buf = [], []
        with torch.no_grad():
            for u, v in dloader_vl:
                u = u.to(DEV)
                y_hat = net(u).cpu()
                preds_buf.append(y_hat)
                labs_buf.append(v)

        preds_val = torch.cat(preds_buf).numpy()
        labs_val = torch.cat(labs_buf).numpy()
        rmse_now = np.sqrt(mean_squared_error(labs_val, preds_val))
        sched.step(rmse_now)

        if rmse_now < lowest_rmse:
            lowest_rmse = rmse_now
            save_best = net.state_dict()
            bad_rounds = 0
        else:
            bad_rounds += 1
            if bad_rounds >= max_wait:
                print("Early stop:", e)
                break

    if save_best:
        net.load_state_dict(save_best)
    return net, lowest_rmse


        
def predict(
    X, Y, groups, model_class, name_prefix="guided", batch_size=64, epochs=60, verbose=True
):

    scaler_Y = StandardScaler()
    Y_scaled = scaler_Y.fit_transform(Y)

    full_data = EMGData(X, Y_scaled)
    logo = LeaveOneGroupOut()

    y_true_all = []
    y_pred_all = []
    rmse_folds = []
    trained_models = []

    os.makedirs("models", exist_ok=True)

    for i, (train_idx, test_idx) in enumerate(logo.split(X, Y_scaled, groups)):
        if verbose:
            print(f"\n Fold {i+1} — Train groups: {np.unique(groups[train_idx])}, Test: {np.unique(groups[test_idx])}")

        tr_dl = DataLoader(Subset(full_data, train_idx), batch_size=batch_size, shuffle=True)
        vl_dl = DataLoader(Subset(full_data, test_idx), batch_size=batch_size, shuffle=False)

        model = model_class()
        model, rmse = execute_training(model, tr_dl, vl_dl, E=epochs)
        rmse_folds.append(rmse)
        trained_models.append(model)

        
        model_path = f"models/{name_prefix}_fold{i+1}.pt"
        torch.save(model.state_dict(), model_path)

        model.eval()
        with torch.no_grad():
            x_test = torch.tensor(X[test_idx], dtype=torch.float32).to(DEV)
            y_pred_scaled = model(x_test).cpu().numpy()

        y_pred = scaler_Y.inverse_transform(y_pred_scaled)
        y_true = Y[test_idx]

        y_pred_all.append(y_pred)
        y_true_all.append(y_true)

    y_pred_all = np.vstack(y_pred_all)
    y_true_all = np.vstack(y_true_all)

 
    dump(scaler_Y, f"models/{name_prefix}_scaler_Y.joblib")

    if verbose:
        rmse_total = np.sqrt(mean_squared_error(y_true_all, y_pred_all))
        r2_total = r2_score(y_true_all, y_pred_all)
        print(f" Global RMSE        : {rmse_total:.4f} | R² : {r2_total:.4f}")

    return y_true_all, y_pred_all, trained_models, scaler_Y


In [11]:
y_true_guided, y_pred_guided, models_guided, scaler_guided = predict(
    X, Y, groups, Net, name_prefix="guided"
)

y_true_free, y_pred_free, models_free, scaler_free = predict(
    X_free, Y_free, groups_free, Net, name_prefix="free"
)



 Fold 1 — Train groups: [1 2 3 4], Test: [0]
Early stop: 29

 Fold 2 — Train groups: [0 2 3 4], Test: [1]
Early stop: 15

 Fold 3 — Train groups: [0 1 3 4], Test: [2]
Early stop: 35

 Fold 4 — Train groups: [0 1 2 4], Test: [3]
Early stop: 39

 Fold 5 — Train groups: [0 1 2 3], Test: [4]
Early stop: 35
 Global RMSE        : 4.5091 | R² : 0.6986

 Fold 1 — Train groups: [1 2 3 4], Test: [0]
Early stop: 23

 Fold 2 — Train groups: [0 2 3 4], Test: [1]
Early stop: 11

 Fold 3 — Train groups: [0 1 3 4], Test: [2]
Early stop: 30

 Fold 4 — Train groups: [0 1 2 4], Test: [3]
Early stop: 28

 Fold 5 — Train groups: [0 1 2 3], Test: [4]
Early stop: 24
 Global RMSE        : 10.7066 | R² : 0.1537


In [ ]:
gs = joblib.load("pkl/gridsearch_results.pkl")
results = pd.DataFrame(gs.cv_results_)
results['RMSE'] = -results['mean_test_score']

gs = joblib.load("pkl/gridsearch_results_free.pkl")
results_free = pd.DataFrame(gs.cv_results_)
results_free['RMSE'] = -results_free['mean_test_score']

def get_best_model_params(results_df, model_type):
    return results_df[results_df['params'].apply(lambda p: isinstance(p['model'], model_type))] \
                     .sort_values('RMSE') \
                     .iloc[0]['params']


best_tree_params_guided = get_best_model_params(results, DecisionTreeRegressor)
best_rf_params_guided  = get_best_model_params(results, RandomForestRegressor)
best_ridge_params_guided    = get_best_model_params(results, Ridge)

best_tree_params_free = get_best_model_params(results_free, DecisionTreeRegressor)
best_rf_params_free  = get_best_model_params(results_free, RandomForestRegressor)
best_ridge_params_free    = get_best_model_params(results_free, Ridge)

def build_model_pipeline(model_params):

    model = model_params['model']
    for key, value in model_params.items():
        if key.startswith('model__'):
            setattr(model, key.split('__')[1], value)

    pipe = make_pipeline(
        Pipeline([
            ('feat', FeatureExtractor()),
            ('pca', PCA(n_components=0.95)),
            ('scale', StandardScaler())
        ]),
        model
    )
    
    return pipe

tree_pipe_guided = build_model_pipeline(best_tree_params_guided).fit(X, Y)
rf_pipe_guided   = build_model_pipeline(best_rf_params_guided).fit(X, Y)
ride_pipe_guided = build_model_pipeline(best_ridge_params_guided).fit(X, Y)

tree_pipe_free   = build_model_pipeline(best_tree_params_free).fit(X_free, Y_free)
rf_pipe_free     = build_model_pipeline(best_rf_params_free).fit(X_free, Y_free)
ride_pipe_free   = build_model_pipeline(best_ridge_params_free).fit(X_free,Y_free)

## Average

In [ ]:
GuidedTest_processed = GuidedTest.reshape(-1, 8, 500)

tree_pred_test = tree_pipe_guided.predict(GuidedTest_processed)
rf_pred_test   = rf_pipe_guided.predict(GuidedTest_processed)

nn_preds_all_models = []
with torch.no_grad():
    x_tensor_test = torch.tensor(GuidedTest_processed, dtype=torch.float32).to(DEV)
    for model in models_guided:
        model.eval()
        pred = model(x_tensor_test).cpu().numpy()
        nn_preds_all_models.append(pred)
        
nn_pred_test = np.mean(np.stack(nn_preds_all_models, axis=0), axis=0)

pred_test_avg = np.mean(np.stack([tree_pred_test, rf_pred_test, nn_pred_test], axis=0), axis=0)

pd.DataFrame(pred_test_avg).to_csv("submission_guided_avg.csv", index=False, header=False)


In [ ]:
FreeTest_processed = FreeTest.reshape(-1, 8, 500)  

tree_pred_test_free = tree_pipe_free.predict(FreeTest_processed)
rf_pred_test_free   = rf_pipe_free.predict(FreeTest_processed)

nn_preds_all_models_free = []
with torch.no_grad():
    x_tensor_free = torch.tensor(FreeTest_processed, dtype=torch.float32).to(DEV)
    for model in models_free:
        model.eval()
        pred = model(x_tensor_free).cpu().numpy()
        nn_preds_all_models_free.append(pred)

nn_pred_test_free = np.mean(np.stack(nn_preds_all_models_free, axis=0), axis=0)

pred_test_avg_free = np.mean(np.stack([tree_pred_test_free, rf_pred_test_free, nn_pred_test_free], axis=0), axis=0)

pd.DataFrame(pred_test_avg_free).to_csv("submission_free_avg.csv", index=False, header=False)


In [ ]:
guided_df = pd.read_csv("submission_guided_avg.csv", header=None)
free_df   = pd.read_csv("submission_free_avg.csv", header=None)

assert guided_df.shape == (1660, 51), f"Expected (1660, 51), got {guided_df.shape}"
assert free_df.shape == (1540, 51), f"Expected (1540, 51), got {free_df.shape}"

final_predictions = pd.concat([guided_df, free_df], axis=0)

assert final_predictions.shape == (3200, 51), "Erreur taille"

final_predictions.to_csv("team_submission.csv", index=False, header=False)

## Meta-learner

We used Ridge regression as meta-learner, trained on the concatenated predictions of the base models. This choice ensures robustness to correlated features and avoids overfitting.

Inputs to the meta-learner were shaped as [n_samples, 4 × 51]. The meta-learner was trained and validated using LOGO cross-validation to ensure fairness and avoid overfitting.

The meta-learner was evaluated using Leave-One-Session-Out CV, ensuring no data leakage between overlapping windows.

In [ ]:
tree_pred_guided = tree_pipe_guided.predict(X)
rf_pred_guided   = rf_pipe_guided.predict(X)
ride_pred_guided = ride_pipe_guided.predict(X)

tree_pred_free   = tree_pipe_free.predict(X_free)
rf_pred_free     = rf_pipe_free.predict(X_free)
ride_pred_free   = ride_pipe_free.predict(X_free)

GuidedTest_processed = GuidedTest.reshape(-1, 8, 500)
FreeTest_processed = FreeTest.reshape(-1, 8, 500)  

tree_pred_test_guided = tree_pipe_guided.predict(GuidedTest_processed)
rf_pred_test_guided   = rf_pipe_guided.predict(GuidedTest_processed)
ride_pred_test_guided = ride_pipe_guided.predict(GuidedTest_processed)
with torch.no_grad():
    x_tensor_guided = torch.tensor(GuidedTest_processed, dtype=torch.float32).to(DEV)
    preds = [model(x_tensor_guided).cpu().numpy() for model in models_guided]
    preds = np.stack(preds, axis=0)
    nn_pred_test_guided = np.mean(preds, axis=0)
    nn_pred_test_guided = scaler_guided.inverse_transform(nn_pred_test_guided)

tree_pred_test_free   = tree_pipe_free.predict(FreeTest_processed)
rf_pred_test_free     = rf_pipe_free.predict(FreeTest_processed)
ride_pred_test_free   = ride_pipe_free.predict(FreeTest_processed)
with torch.no_grad():
    x_tensor_guided = torch.tensor(FreeTest_processed, dtype=torch.float32).to(DEV)
    preds = [model(x_tensor_guided).cpu().numpy() for model in models_free]
    preds = np.stack(preds, axis=0)
    nn_pred_test_free = np.mean(preds, axis=0)
    nn_pred_test_free = scaler_free.inverse_transform(nn_pred_test_free)

In [ ]:
Mix_guided = np.stack([tree_pred_guided, rf_pred_guided, ride_pred_guided, y_pred_guided], axis=1)
Mix_free   = np.stack([tree_pred_free, rf_pred_free, ride_pred_free, y_pred_free], axis=1) 

groups_meta = []
for i in range (5):
    for j in range(918):
        groups_meta.append(i)

groups_meta = np.array(groups_meta)

In [ ]:
# Guided data
X_guided_meta = Mix_guided.reshape(Mix_guided.shape[0], -1)

logo = LeaveOneGroupOut()
preds_guided, targets_guided, fold_rmse_guided = [], [], []

for fold_idx, (train_idx, val_idx) in enumerate(logo.split(X_guided_meta, Y, groups_meta)):
    meta = Ridge(alpha=1.0)
    meta.fit(X_guided_meta[train_idx], Y[train_idx])
    
    fold_pred = meta.predict(X_guided_meta[val_idx])
    
    preds_guided.append(fold_pred)
    targets_guided.append(Y[val_idx])
    fold_rmse_guided.append(np.sqrt(mean_squared_error(Y[val_idx], fold_pred)))

preds_guided = np.vstack(preds_guided)
targets_guided = np.vstack(targets_guided)

overall_rmse_guided = np.sqrt(mean_squared_error(targets_guided, preds_guided))
r2_guided = r2_score(targets_guided, preds_guided)

print("Guided data")
print("Mean RMSE:", np.mean(fold_rmse_guided))
print("Overall RMSE:", overall_rmse_guided)
print("R² score:", r2_guided)

# Free data
X_free_meta = Mix_free.reshape(Mix_free.shape[0], -1)
group_ids_free = np.repeat(np.arange(5), 918)

preds_free, targets_free, fold_rmse_free = [], [], []

for fold_idx, (train_idx, val_idx) in enumerate(logo.split(X_free_meta, Y_free, group_ids_free)):
    meta = Ridge(alpha=1.0)
    meta.fit(X_free_meta[train_idx], Y_free[train_idx])
    
    fold_pred = meta.predict(X_free_meta[val_idx])
    
    preds_free.append(fold_pred)
    targets_free.append(Y_free[val_idx])
    fold_rmse_free.append(np.sqrt(mean_squared_error(Y_free[val_idx], fold_pred)))

preds_free = np.vstack(preds_free)
targets_free = np.vstack(targets_free)

overall_rmse_free = np.sqrt(mean_squared_error(targets_free, preds_free))
r2_free = r2_score(targets_free, preds_free)

print("Free data")
print("Mean RMSE:", np.mean(fold_rmse_free))
print("Overall RMSE:", overall_rmse_free)
print("R² score:", r2_free)

final_guided_model = Ridge(alpha=1.0).fit(X_guided_meta, Y)
final_free_model = Ridge(alpha=1.0).fit(X_free_meta, Y_free)

X_test_guided = np.stack([tree_pred_test_guided, rf_pred_test_guided, ride_pred_test_guided, nn_pred_test_guided], axis=1).reshape(-1, 4 * 51)
X_test_free   = np.stack([tree_pred_test_free, rf_pred_test_free, ride_pred_test_free, nn_pred_test_free],   axis=1).reshape(-1, 4 * 51)

final_preds_guided = final_guided_model.predict(X_test_guided)
final_preds_free   = final_free_model.predict(X_test_free)

# Combine both parts for submission
final_submission = np.vstack([final_preds_guided, final_preds_free])
pd.DataFrame(final_submission).to_csv("team_submission.csv", index=False, header=False)

In [ ]:
avg_pred_guided = (tree_pred_guided + rf_pred_guided + ride_pred_guided + y_pred_guided) / 4
avg_pred_free   = (tree_pred_free + rf_pred_free + ride_pred_free + y_pred_free) / 4

def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(name, "RMSE: ", rmse, "R^2: ", r2)
    return rmse, r2

print("Performance on GUIDED:")
evaluate("Tree", Y, tree_pred_guided)
evaluate("Random Forest", Y, rf_pred_guided)
evaluate("Ridge", Y, ride_pred_guided)
evaluate("CNN", Y, y_pred_guided)
evaluate("Average Ensemble", Y, avg_pred_guided)
evaluate("Meta-Learner", Y, preds_guided)

print("Performance on FREE:")
evaluate("Tree", Y_free, tree_pred_free)
evaluate("Random Forest", Y_free, rf_pred_free)
evaluate("Ridge", Y_free, ride_pred_free)
evaluate("CNN", Y_free, y_pred_free)
evaluate("Average Ensemble", Y_free, avg_pred_free)
evaluate("Meta-Learner", Y_free, preds_free)

### Performance on GUIDED

| Model             | RMSE     | R²     |
|------------------|----------|--------|
| Tree             | 6.8003   | 0.6364 |
| Random Forest    | 3.6995   | 0.8248 |
| Ridge            | 12.9697  | 0.1529 |
| Average Ensemble | 6.6788   | 0.6697 |
| Meta-Learner     | 3.3521   | 0.7903 |

### Performance on FREE

| Model             | RMSE     | R²     |
|------------------|----------|--------|
| Tree             | 10.5259  | 0.2220 |
| Random Forest    | 7.9711   | 0.5150 |
| Ridge            | 12.2356  | 0.0384 |
| Average Ensemble | 9.7969   | 0.3195 |
| Meta-Learner     | 7.0956   | 0.5615 |


The meta-learner effectively leverages the strengths of Random Forest and CNN while suppressing weaker contributors like Ridge. It shows better generalization on free gestures.

In [ ]:
coefs = final_guided_model.coef_
contributions = np.linalg.norm(coefs.reshape(51, 4, 51), axis=(0, 2))

print("Base model contributions (guided):")
for name, c in zip(["Tree", "RF", "Ridge", "NN"], contributions):
    print(name, c)

coefs = final_free_model.coef_
contributions = np.linalg.norm(coefs.reshape(51, 4, 51), axis=(0, 2))
print("Base model contributions (free):")
for name, c in zip(["Tree", "RF", "Ridge", "NN"], contributions):
    print(name, c)

Base model contributions (guided):
Tree 50.39494374277807
RF 111.76438367043532
Ridge 0.4006383019290114
NN 103.92681137740334
Base model contributions (free):
Tree 90.22833063252224
RF 177.31962822672202
Ridge 1.8106177770613539
NN 148.2930477124431


We computed the L2-norm of the meta-learner’s coefficients for each base model to quantify their influence.
* Random Forest and CNN are the dominant contributors.
* Ridge regression contributes marginally, likely due to its poor standalone performance.
* Tree models contribute moderately, suggesting they still add some complementary structure.